# Qwen Tor Researcher LoRA finetunning notebook
> Github @y3chnx  |   HuggingFace @y3chnx <br>

- This is the Jupyter Notebook that allows you to finetune Qwen based model into darkweb expert. 
<br>
- This Notebook is designed for Kaggle Notebooks. 
- Change Hugging Face repository directory and Hugging Face Token. 

In [ ]:
# 1) Install deps without disturbing pre-installed CUDA/RAPIDS stacks.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


pip("-U", "transformers>=4.51.0", "trl>=0.12.0", "peft>=0.13.0",
    "datasets>=3.0.0", "accelerate>=1.0.0", "huggingface_hub")

pip("--no-deps", "-U", "bitsandbytes>=0.43.0")
print("installs done — restart runtime only if prompted, then continue from Cell 2")

In [ ]:
# 2) Config + HF auth (token entered directly in code)
import os
from huggingface_hub import login

HF_TOKEN   = "YOUR_HF_TOKEN"   # <-- PASTE your HF WRITE token
HF_REPO    = "yourname/qwen-tor-researcher"        # <-- CHANGE to your repo
BASE_MODEL = "Qwen/Qwen3-4B"
DATASET    = "y3chnx/qwen_tor_researcher_dataset"
MAXLEN     = 2048

login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# 3) Load + format dataset into the Qwen chat template
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ds = load_dataset(DATASET, split="train")

def to_text(ex):
    user = ex["instruction"]
    if ex.get("context"):                       
        user += "\n\n[ARTIFACT]\n" + ex["context"]
    msgs = [{"role": "system", "content": ex.get("system", "You are a Tor research assistant.")},
            {"role": "user",   "content": user},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tok.apply_chat_template(msgs, tokenize=False, enable_thinking=False)}

ds = ds.map(to_text, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=0.02, seed=42)
print(ds, "\n--- sample ---\n", ds["train"][0]["text"][:600])

In [ ]:
# 4) Load Qwen3-4B in 4-bit + attach LoRA adapters
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map="auto",
    torch_dtype=torch.bfloat16, trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
# 5) Train (QLoRA) + push checkpoints to the Hub every save
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir="qwen3-4b-tor",
    dataset_text_field="text",
    max_length=1024,                    
    packing=False,                       
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,                       
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=False,        
    report_to="none",
    push_to_hub=True,
    hub_model_id=HF_REPO,
    hub_strategy="every_save",
    hub_token=HF_TOKEN,
)

trainer = SFTTrainer(
    model=model, args=cfg, peft_config=lora,
    train_dataset=ds["train"], eval_dataset=ds["test"], processing_class=tok,
)
trainer.train()

In [ ]:
# 7) Merge LoRA → F16 GGUF Convert → HuggingFace Upload
import subprocess, sys, os, gc, shutil, torch
from huggingface_hub import HfApi, create_repo

ADAPTER_REPO = "yourname/qwen-tor-researcher"
BASE_MODEL   = "Qwen/Qwen3-4B"
MERGED_DIR   = "/kaggle/working/model_export"
GGUF_DIR     = "/kaggle/working/gguf_output"
LLAMA_DIR    = "/kaggle/working/llama.cpp"
HF_GGUF_REPO = "yourname/qwen-tor-researcher-gguf"
GGUF_F16     = f"{GGUF_DIR}/model-f16.gguf"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR,   exist_ok=True)

def run(cmd):
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

def disk():
    s = shutil.disk_usage("/kaggle/working")
    print(f"free: {s.free/1e9:.1f} GB / total: {s.total/1e9:.1f} GB")

# ── Step 1. Package Installation ───────────────────────────────────────────────────
print("[1/5] Installing packages…")
run(f"{sys.executable} -m pip install -q peft transformers accelerate "
    f"huggingface_hub safetensors sentencepiece")

# ── Step 2. LoRA → Full model merge ──────────────────────────────────────
print("\n[2/5] Merging LoRA adapter…")
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16,
    device_map="cpu", trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tok.save_pretrained(MERGED_DIR)
print(f"  → Saved to: {MERGED_DIR}")

del model, base
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
disk()

# ── Step 3. llama.cpp clone ───────────────────────────────────────────────
print("\n[3/5] Cloning llama.cpp…")
if os.path.isdir(LLAMA_DIR):
    run(f"rm -rf {LLAMA_DIR}")
run(f"git clone https://github.com/ggerganov/llama.cpp {LLAMA_DIR} -q")
run(f"{sys.executable} -m pip install -q -r {LLAMA_DIR}/requirements.txt")

# ── Step 4. F16 GGUF Converting ────────────────────────────────────────────────
print("\n[4/5] Converting to F16 GGUF…")
run(f"{sys.executable} {LLAMA_DIR}/convert_hf_to_gguf.py {MERGED_DIR} "
    f"--outfile {GGUF_F16} --outtype f16")
print("  → F16 GGUF done")


shutil.rmtree(MERGED_DIR)
print("deleted merged safetensors")
disk()

# ── Step 5. Upload ────────────────────────────────────────────
print("\n[5/5] Uploading to HuggingFace Hub…")
api = HfApi(token=HF_TOKEN)
create_repo(HF_GGUF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)

fname = os.path.basename(GGUF_F16)
size  = os.path.getsize(GGUF_F16) / 1e9
print(f"  Uploading {fname} ({size:.2f} GB)…")
api.upload_file(
    path_or_fileobj=GGUF_F16,
    path_in_repo=fname,
    repo_id=HF_GGUF_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"  ✓ {fname} uploaded!")

print(f"\nSuccess! → https://huggingface.co/{HF_GGUF_REPO}")